# Pre-Entrega 2 — Adquisición e integración de datos de ventas

**Curso:** Data Science II · Coderhouse
**Autor:** Tomás Rotllant

## Objetivo
Simular un escenario real en el que los datos de ventas vienen de **dos departamentos con formatos distintos**:

| Fuente | Formato | Columnas |
|---|---|---|
| Transacciones | CSV | `id_transaccion`, `id_producto`, `cantidad`, `fecha` |
| Maestro de productos | Excel (.xlsx) | `id_producto`, `nombre_producto`, `precio_unitario` |

Con `pandas` se integran ambas fuentes en un dataset limpio, con la columna calculada `total_venta`, y se exporta a **Parquet**.

## Contenido
0. Configuración del entorno
1. Módulo ETL (`src/etl.py`)
2. Adquisición de datos (CSV + Excel)
3. Transformación: tipos de datos y valores nulos
4. Merge por `id_producto`
5. Normalización: columna `total_venta`
6. Validación del dataset final
7. Exportación a Parquet

**Abrir en Colab:** *Entorno de ejecución → Ejecutar todas*. El notebook genera los datos, el módulo `src/etl.py` y el Parquet final.

## 0. Configuración del entorno
Instalación de dependencias, parámetros del pipeline y estructura de carpetas del proyecto.

In [1]:
# Colab ya trae pandas y numpy; se fuerzan openpyxl (lectura/escritura de Excel)
# y pyarrow (motor de Parquet) para que el notebook sea reproducible en cualquier entorno.
%pip install -q "pandas>=2.0" "numpy>=1.24" "openpyxl>=3.1" "pyarrow>=14"

In [2]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# ---------------- Parámetros del pipeline ----------------
SEED = 42                 # semilla -> los datos generados son siempre los mismos
N_TRANSACCIONES = 1_000

# ---------------- Rutas del proyecto ----------------
IN_COLAB = "google.colab" in sys.modules
PROJECT_DIR = Path("/content/pre-entrega-2") if IN_COLAB else Path.cwd() / "pre-entrega-2"
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_DIR)     # todas las rutas siguientes son relativas a la raíz del proyecto

RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")
for carpeta in (RAW_DIR, PROCESSED_DIR, Path("src")):
    carpeta.mkdir(parents=True, exist_ok=True)

PATH_CSV = RAW_DIR / "transacciones.csv"
PATH_XLSX = RAW_DIR / "maestro_productos.xlsx"
PATH_PARQUET = PROCESSED_DIR / "ventas_consolidadas.parquet"

pd.set_option("display.max_columns", 20)
print(f"Carpeta del proyecto: {PROJECT_DIR.name}/")
print(f"pandas {pd.__version__} | numpy {np.__version__}")

Carpeta del proyecto: pre-entrega-2/
pandas 2.2.3 | numpy 2.1.3


## 1. Módulo ETL

Toda la lógica vive en funciones dentro de `src/etl.py` y el notebook solo las **orquesta y muestra resultados**. Esto permite:
- **Modularización y reutilización:** cada paso (cargar, limpiar, integrar, validar, exportar) es una función con una sola responsabilidad.
- **Reproducibilidad fuera de Colab:** `python src/etl.py` corre el pipeline completo sin abrir el notebook.
- **Trazabilidad:** cada función de limpieza devuelve un *reporte* con cuántas filas se eliminaron o imputaron y por qué.

In [3]:
%%writefile src/etl.py
"""ETL de ventas: integra transacciones (CSV) con el maestro de productos (Excel).

Pasos: adquisición -> tipado -> tratamiento de nulos -> merge -> total_venta
-> validación -> exportación a Parquet.

Uso fuera del notebook:  python src/etl.py
"""
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

FORMATO_FECHA = "%d/%m/%Y"
COLUMNAS_CRITICAS = [
    "id_transaccion", "fecha", "id_producto", "cantidad", "precio_unitario", "total_venta",
]
ORDEN_FINAL = [
    "id_transaccion", "fecha", "id_producto", "nombre_producto",
    "cantidad", "precio_unitario", "total_venta",
]


# --------------------------------------------------------------------------
# 1) Generación de datos crudos (simula las exportaciones de cada departamento)
# --------------------------------------------------------------------------
def generar_datos_crudos(
    path_csv: Path, path_xlsx: Path, n_transacciones: int = 1_000, seed: int = 42
) -> None:
    """Crea un CSV de transacciones y un Excel de productos con problemas típicos.

    Problemas introducidos a propósito (para luego tratarlos):
      CSV  -> IDs como texto con espacios y ceros a la izquierda, IDs nulos,
              IDs inexistentes en el maestro, cantidades nulas o <= 0,
              fechas nulas y fechas imposibles (31/02).
      XLSX -> nombre de producto nulo, precio nulo, ID de producto duplicado.
    """
    rng = np.random.default_rng(seed)

    # ---- Maestro de productos (Excel, departamento de Compras) ----
    nombres = [
        "Notebook 14''", "Mouse inalámbrico", "Teclado mecánico", "Monitor 24''",
        "Auriculares Bluetooth", "Webcam HD", "Disco SSD 1TB", "Memoria RAM 16GB",
        "Router WiFi 6", "Impresora láser", "Silla ergonómica", "Escritorio regulable",
        "Hub USB-C", "Parlante portátil", "Tablet 10''", "Smartwatch",
        "Cable HDMI 2m", "Pendrive 64GB", "Lámpara LED", "Mochila para notebook",
    ]
    ids = np.arange(101, 101 + len(nombres))
    productos = pd.DataFrame({
        "id_producto": ids,
        "nombre_producto": nombres,
        "precio_unitario": rng.uniform(5_000, 900_000, len(ids)).round(2),
    })
    productos["nombre_producto"] = productos["nombre_producto"].astype(object)
    productos.loc[3, "nombre_producto"] = np.nan          # nombre faltante (no crítico)
    productos.loc[7, "precio_unitario"] = np.nan          # precio faltante (crítico)
    productos = pd.concat([productos, productos.iloc[[10]]], ignore_index=True)  # ID duplicado

    # ---- Transacciones (CSV, departamento de Ventas) ----
    fechas = pd.Timestamp("2026-01-01") + pd.to_timedelta(
        rng.integers(0, 300, n_transacciones), unit="D"
    )
    trans = pd.DataFrame({
        "id_transaccion": np.arange(1, n_transacciones + 1),
        "id_producto": rng.choice(ids, n_transacciones).astype(str).astype(object),
        "cantidad": rng.integers(1, 11, n_transacciones).astype(float),
        "fecha": fechas.strftime(FORMATO_FECHA).astype(object),
    })

    # Se reparten los problemas en filas distintas (sin superposición)
    idx = rng.permutation(n_transacciones)
    bloques = np.split(idx[:140], [40, 55, 65, 80, 90, 100, 110])
    con_espacios, id_nulo, id_huerfano, cant_nula, cant_no_positiva, fecha_nula, fecha_invalida, ceros = bloques

    trans.loc[con_espacios, "id_producto"] = "  " + trans.loc[con_espacios, "id_producto"] + " "
    trans.loc[ceros, "id_producto"] = "0" + trans.loc[ceros, "id_producto"]
    trans.loc[id_nulo, "id_producto"] = np.nan
    trans.loc[id_huerfano, "id_producto"] = "999"          # no existe en el maestro
    trans.loc[cant_nula, "cantidad"] = np.nan
    trans.loc[cant_no_positiva, "cantidad"] = rng.choice([0, -1, -2], len(cant_no_positiva))
    trans.loc[fecha_nula, "fecha"] = np.nan
    trans.loc[fecha_invalida, "fecha"] = "31/02/2026"     # fecha imposible

    path_csv.parent.mkdir(parents=True, exist_ok=True)
    trans.to_csv(path_csv, index=False)

    # Context manager: el ExcelWriter (openpyxl) se cierra siempre, aunque haya error
    with pd.ExcelWriter(path_xlsx, engine="openpyxl") as writer:
        productos.to_excel(writer, sheet_name="productos", index=False)


# --------------------------------------------------------------------------
# 2) Adquisición
# --------------------------------------------------------------------------
def cargar_transacciones(path_csv: Path) -> pd.DataFrame:
    """Lee el CSV forzando id_producto y fecha como texto.

    Leer las columnas 'sucias' como texto evita que pandas infiera tipos
    distintos según el contenido; la conversión se hace explícitamente después.
    """
    return pd.read_csv(path_csv, dtype={"id_producto": "string", "fecha": "string"})


def cargar_productos(path_xlsx: Path, hoja: str = "productos") -> pd.DataFrame:
    """Lee el Excel con openpyxl dentro de un context manager (cierra el archivo)."""
    with pd.ExcelFile(path_xlsx, engine="openpyxl") as xls:
        return pd.read_excel(xls, sheet_name=hoja)


# --------------------------------------------------------------------------
# 3) Transformación
# --------------------------------------------------------------------------
def normalizar_id(serie: pd.Series) -> pd.Series:
    """Texto -> entero nullable (Int64): quita espacios y ceros a la izquierda.

    Un valor no numérico queda como <NA> en lugar de cortar la ejecución.
    """
    limpio = serie.astype("string").str.strip()
    return pd.to_numeric(limpio, errors="coerce").astype("Int64")


def limpiar_transacciones(df: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """Tipa las columnas y elimina filas sin datos críticos.

    Criterio: id_producto, fecha y cantidad son críticos. Imputarlos sería
    inventar ventas (no hay forma de saber qué producto, cuándo o cuánto se
    vendió), así que esas filas se eliminan y se informa cuántas fueron.
    """
    df = df.copy()
    reporte: dict = {"filas_iniciales": len(df)}

    fecha_texto = df["fecha"].astype("string").str.strip()
    df["id_producto"] = normalizar_id(df["id_producto"])
    df["fecha"] = pd.to_datetime(fecha_texto, format=FORMATO_FECHA, errors="coerce")
    df["cantidad"] = pd.to_numeric(df["cantidad"], errors="coerce")

    reporte["fechas_con_formato_invalido"] = int((fecha_texto.notna() & df["fecha"].isna()).sum())
    reporte["nulos_tras_tipado"] = df[["id_producto", "fecha", "cantidad"]].isna().sum().to_dict()

    df = df.dropna(subset=["id_producto", "fecha", "cantidad"])
    reporte["eliminadas_por_nulos"] = reporte["filas_iniciales"] - len(df)

    no_positivas = df["cantidad"] <= 0
    reporte["eliminadas_por_cantidad_no_positiva"] = int(no_positivas.sum())
    df = df.loc[~no_positivas]

    df = df.astype({"id_transaccion": "int64", "id_producto": "int64", "cantidad": "int64"})
    reporte["filas_finales"] = len(df)
    return df.reset_index(drop=True), reporte


def limpiar_productos(df: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """Tipa el maestro, imputa lo no crítico y elimina lo que no se puede usar.

    - nombre_producto (descriptivo, no crítico) -> se imputa "Sin nombre".
    - precio_unitario (crítico para total_venta) -> se elimina el producto:
      imputar un precio (p. ej. la media) inventaría facturación.
    - id_producto duplicado -> se conserva la primera aparición, para que el
      merge sea muchos-a-uno y no duplique transacciones.
    """
    df = df.copy()
    reporte: dict = {"filas_iniciales": len(df)}

    df["id_producto"] = normalizar_id(df["id_producto"])
    df["precio_unitario"] = pd.to_numeric(df["precio_unitario"], errors="coerce").astype("float64")
    nombres = df["nombre_producto"].astype("string").str.strip()

    reporte["nombres_imputados"] = int(nombres.isna().sum())
    df["nombre_producto"] = nombres.fillna("Sin nombre")

    invalidos = df["id_producto"].isna() | df["precio_unitario"].isna() | (df["precio_unitario"] <= 0)
    reporte["productos_eliminados_sin_id_o_precio"] = df.loc[invalidos, "id_producto"].tolist()
    df = df.loc[~invalidos]

    duplicados = df["id_producto"].duplicated(keep="first")
    reporte["ids_duplicados_eliminados"] = df.loc[duplicados, "id_producto"].tolist()
    df = df.loc[~duplicados]

    df = df.astype({"id_producto": "int64"})
    reporte["filas_finales"] = len(df)
    return df.reset_index(drop=True), reporte


# --------------------------------------------------------------------------
# 4) Merge
# --------------------------------------------------------------------------
def integrar(trans: pd.DataFrame, prod: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """Left join transacciones -> productos por id_producto.

    - Verifica que la llave tenga el mismo tipo en ambos lados.
    - validate="many_to_one" corta la ejecución si el maestro tuviera IDs repetidos.
    - indicator=True permite contar las transacciones sin producto en el maestro
      (huérfanas): se informan y se excluyen porque no tienen precio.
    """
    tipo_t, tipo_p = trans["id_producto"].dtype, prod["id_producto"].dtype
    if tipo_t != tipo_p:
        raise TypeError(f"Llave con tipos distintos: transacciones={tipo_t}, productos={tipo_p}")

    df = trans.merge(prod, on="id_producto", how="left", validate="many_to_one", indicator=True)
    huerfanas = df["_merge"].eq("left_only")
    reporte = {
        "filas_tras_merge": len(df),
        "transacciones_huerfanas": int(huerfanas.sum()),
        "ids_huerfanos": sorted(df.loc[huerfanas, "id_producto"].unique().tolist()),
    }
    df = df.loc[~huerfanas].drop(columns="_merge").reset_index(drop=True)
    reporte["filas_finales"] = len(df)
    return df, reporte


# --------------------------------------------------------------------------
# 5) Normalización
# --------------------------------------------------------------------------
def agregar_total_venta(df: pd.DataFrame) -> pd.DataFrame:
    """total_venta = cantidad * precio_unitario (redondeado a 2 decimales)."""
    df = df.assign(total_venta=(df["cantidad"] * df["precio_unitario"]).round(2))
    return df[ORDEN_FINAL].sort_values(["fecha", "id_transaccion"]).reset_index(drop=True)


# --------------------------------------------------------------------------
# 6) Validación
# --------------------------------------------------------------------------
def validar(df: pd.DataFrame) -> None:
    """Chequeos de calidad del dataset final. Lanza ValueError si alguno falla."""
    nulos = df[COLUMNAS_CRITICAS].isna().sum()
    if nulos.any():
        raise ValueError(f"Hay nulos en campos críticos:\n{nulos[nulos > 0]}")
    if not pd.api.types.is_datetime64_any_dtype(df["fecha"]):
        raise ValueError("La columna fecha no es datetime")
    if not df["id_transaccion"].is_unique:
        raise ValueError("Hay id_transaccion duplicados (el merge duplicó filas)")
    if (df["cantidad"] <= 0).any() or (df["precio_unitario"] <= 0).any():
        raise ValueError("Hay cantidades o precios no positivos")
    esperado = (df["cantidad"] * df["precio_unitario"]).round(2)
    if not np.allclose(df["total_venta"], esperado):
        raise ValueError("total_venta no coincide con cantidad * precio_unitario")


# --------------------------------------------------------------------------
# 7) Exportación
# --------------------------------------------------------------------------
def exportar_parquet(df: pd.DataFrame, path: Path) -> Path:
    """Guarda en Parquet (columnar, comprimido y conserva los tipos)."""
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(path, index=False, engine="pyarrow")
    return path


# --------------------------------------------------------------------------
# Pipeline completo
# --------------------------------------------------------------------------
def ejecutar_pipeline(base_dir: Path = Path("."), n_transacciones: int = 1_000, seed: int = 42) -> pd.DataFrame:
    base_dir = Path(base_dir)
    path_csv = base_dir / "data/raw/transacciones.csv"
    path_xlsx = base_dir / "data/raw/maestro_productos.xlsx"
    path_parquet = base_dir / "data/processed/ventas_consolidadas.parquet"

    generar_datos_crudos(path_csv, path_xlsx, n_transacciones, seed)
    trans, rep_t = limpiar_transacciones(cargar_transacciones(path_csv))
    prod, rep_p = limpiar_productos(cargar_productos(path_xlsx))
    df, rep_m = integrar(trans, prod)
    df = agregar_total_venta(df)
    validar(df)
    exportar_parquet(df, path_parquet)

    for nombre, rep in (("Transacciones", rep_t), ("Productos", rep_p), ("Merge", rep_m)):
        print(f"{nombre}: {rep}")
    print(f"Dataset final: {df.shape} -> {path_parquet}")
    return df


if __name__ == "__main__":
    # Permite correr el pipeline desde la raíz del repo: python src/etl.py
    ejecutar_pipeline(Path(__file__).resolve().parents[1])

Writing src/etl.py


In [4]:
# Se importa el módulo recién escrito. reload() asegura usar la última versión
# si la celda anterior se vuelve a ejecutar con cambios.
import importlib

sys.path.insert(0, str(Path("src").resolve()))
import etl
importlib.reload(etl)
print("Módulo etl cargado:", [f for f in dir(etl) if not f.startswith("_") and callable(getattr(etl, f))])

Módulo etl cargado: ['Path', 'agregar_total_venta', 'cargar_productos', 'cargar_transacciones', 'ejecutar_pipeline', 'exportar_parquet', 'generar_datos_crudos', 'integrar', 'limpiar_productos', 'limpiar_transacciones', 'normalizar_id', 'validar']


## 2. Adquisición de datos

Se generan las dos fuentes con una semilla fija (`SEED`), así cualquiera que corra el notebook obtiene exactamente los mismos archivos. Para que la limpieza tenga sentido, los datos incluyen **errores típicos de una exportación real**:

| Fuente | Problema introducido |
|---|---|
| CSV | `id_producto` como texto, con espacios y ceros a la izquierda (`"  105 "`, `"0105"`) |
| CSV | `id_producto` nulo y un ID (`999`) que no existe en el maestro |
| CSV | `cantidad` nula, en 0 o negativa |
| CSV | `fecha` nula y fechas imposibles (`31/02/2026`) |
| Excel | `nombre_producto` nulo, `precio_unitario` nulo y un `id_producto` duplicado |

**Lectura:**
- CSV → `id_producto` y `fecha` se leen como texto a propósito, para controlar la conversión en vez de depender de la inferencia de pandas.
- Excel → `pd.ExcelFile(..., engine="openpyxl")` dentro de un `with`: el archivo se cierra siempre al terminar la lectura (error común de la consigna).

In [5]:
etl.generar_datos_crudos(PATH_CSV, PATH_XLSX, n_transacciones=N_TRANSACCIONES, seed=SEED)

trans_raw = etl.cargar_transacciones(PATH_CSV)
prod_raw = etl.cargar_productos(PATH_XLSX)

print(f"Transacciones (CSV): {trans_raw.shape}")
display(trans_raw.head())
print(f"\nMaestro de productos (Excel): {prod_raw.shape}")
display(prod_raw.head())

Transacciones (CSV): (1000, 4)


,id_transaccion,id_producto,cantidad,fecha
0,1,115,3.0,19/02/2026
1,2,118,9.0,16/08/2026
2,3,109,8.0,30/07/2026
3,4,108,8.0,17/04/2026
4,5,115,3.0,21/01/2026



Maestro de productos (Excel): (21, 3)


,id_producto,nombre_producto,precio_unitario
0,101,Notebook 14'',697690.66
1,102,Mouse inalámbrico,397796.20
2,103,Teclado mecánico,773445.14
3,104,NaN,629144.39
4,105,Auriculares Bluetooth,89288.73


In [6]:
# El problema del merge en una línea: la misma llave tiene tipos distintos en cada fuente
print("Tipo de id_producto en el CSV  :", trans_raw["id_producto"].dtype)
print("Tipo de id_producto en el Excel:", prod_raw["id_producto"].dtype)
print("\nEjemplos de IDs crudos en el CSV:", trans_raw["id_producto"].dropna().loc[lambda s: s.str.len() != 3].unique()[:6].tolist())

print("\nTipos de datos crudos (CSV):")
print(trans_raw.dtypes)

Tipo de id_producto en el CSV  : string
Tipo de id_producto en el Excel: int64

Ejemplos de IDs crudos en el CSV: ['  115 ', '0111', '0116', '0109', '0103', '0104']

Tipos de datos crudos (CSV):
id_transaccion             int64
id_producto       string[python]
cantidad                 float64
fecha             string[python]
dtype: object


## 3. Transformación: tipos de datos y valores nulos

### Diagnóstico inicial

In [7]:
def resumen_nulos(df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        "nulos": df.isna().sum(),
        "% nulos": (df.isna().mean() * 100).round(2),
        "tipo": df.dtypes.astype(str),
    })

print("Transacciones (crudo)")
display(resumen_nulos(trans_raw))
print("Maestro de productos (crudo)")
display(resumen_nulos(prod_raw))
print("IDs duplicados en el maestro:", prod_raw.loc[prod_raw["id_producto"].duplicated(), "id_producto"].tolist())

Transacciones (crudo)


,nulos,% nulos,tipo
id_transaccion,0,0.0,int64
id_producto,15,1.5,string
cantidad,15,1.5,float64
fecha,10,1.0,string


Maestro de productos (crudo)


,nulos,% nulos,tipo
id_producto,0,0.00,int64
nombre_producto,1,4.76,object
precio_unitario,1,4.76,float64


IDs duplicados en el maestro: [111]


### Decisiones de limpieza

| Columna | Tratamiento | Justificación |
|---|---|---|
| `id_producto` (ambas fuentes) | `strip` + `to_numeric` → entero | Unifica el tipo de la llave; `"  105 "`, `"0105"` y `105` pasan a ser el mismo ID |
| `fecha` | `to_datetime(format="%d/%m/%Y", errors="coerce")` | Formato explícito (evita confundir día/mes); las fechas imposibles quedan como `NaT` en vez de cortar la ejecución |
| `id_producto`, `fecha`, `cantidad` nulos | **Eliminar** la fila | Son críticos: imputarlos sería inventar una venta |
| `cantidad <= 0` | **Eliminar** la fila | No es una venta válida (posibles devoluciones o errores de carga) |
| `nombre_producto` nulo | **Imputar** `"Sin nombre"` | Es descriptivo: no afecta cálculos y no conviene perder el precio del producto |
| `precio_unitario` nulo | **Eliminar** el producto | Sin precio no se puede calcular `total_venta`; imputar la media distorsionaría la facturación |
| `id_producto` duplicado en el maestro | Conservar la primera aparición | Garantiza un merge muchos-a-uno (si no, se duplican transacciones) |

> **Nota sobre data leakage:** acá no hay modelo, pero se evitó a propósito imputar con estadísticos del dataset (media/mediana). Si más adelante se entrena un modelo, ese tipo de imputación debe calcularse solo con el set de entrenamiento.

In [8]:
trans, rep_trans = etl.limpiar_transacciones(trans_raw)

print("Reporte de limpieza — Transacciones")
for k, v in rep_trans.items():
    print(f"  {k}: {v}")
print("\nTipos tras la limpieza:")
print(trans.dtypes)

Reporte de limpieza — Transacciones
  filas_iniciales: 1000
  fechas_con_formato_invalido: 10
  nulos_tras_tipado: {'id_producto': 15, 'fecha': 20, 'cantidad': 15}
  eliminadas_por_nulos: 50
  eliminadas_por_cantidad_no_positiva: 10
  filas_finales: 940

Tipos tras la limpieza:
id_transaccion             int64
id_producto                int64
cantidad                   int64
fecha             datetime64[ns]
dtype: object


In [9]:
prod, rep_prod = etl.limpiar_productos(prod_raw)

print("Reporte de limpieza — Maestro de productos")
for k, v in rep_prod.items():
    print(f"  {k}: {v}")
print("\nTipos tras la limpieza:")
print(prod.dtypes)
display(prod.head())

Reporte de limpieza — Maestro de productos
  filas_iniciales: 21
  nombres_imputados: 1
  productos_eliminados_sin_id_o_precio: [108]
  ids_duplicados_eliminados: [111]
  filas_finales: 19

Tipos tras la limpieza:
id_producto                 int64
nombre_producto    string[python]
precio_unitario           float64
dtype: object


,id_producto,nombre_producto,precio_unitario
0,101,Notebook 14'',697690.66
1,102,Mouse inalámbrico,397796.20
2,103,Teclado mecánico,773445.14
3,104,Sin nombre,629144.39
4,105,Auriculares Bluetooth,89288.73


## 4. Merge entre fuentes

- **Llave:** `id_producto`, ya convertida a `int64` en ambos DataFrames (la función verifica que los tipos coincidan antes de unir).
- **Tipo de join:** `left` desde transacciones, con `indicator=True` para **medir** cuántas ventas no encuentran su producto en el maestro, en vez de que desaparezcan sin aviso como pasaría con un `inner`.
- **`validate="many_to_one"`:** pandas corta la ejecución si el maestro tuviera IDs repetidos, lo que duplicaría ventas.
- Las transacciones huérfanas (ID `999` y las del producto sin precio) se informan y se excluyen, porque no tienen precio para calcular el total.

In [10]:
ventas, rep_merge = etl.integrar(trans, prod)

print("Reporte del merge")
for k, v in rep_merge.items():
    print(f"  {k}: {v}")
display(ventas.head())

Reporte del merge
  filas_tras_merge: 940
  transacciones_huerfanas: 54
  ids_huerfanos: [108, 999]
  filas_finales: 886


,id_transaccion,id_producto,cantidad,fecha,nombre_producto,precio_unitario
0,1,115,3,2026-02-19,Tablet 10'',401855.71
1,2,118,9,2026-08-16,Pendrive 64GB,62116.44
2,3,109,8,2026-07-30,Router WiFi 6,119661.70
3,5,115,3,2026-01-21,Tablet 10'',401855.71
4,6,104,10,2026-10-19,Sin nombre,629144.39


## 5. Normalización: columna calculada `total_venta`

$$\text{total\_venta} = \text{cantidad} \times \text{precio\_unitario}$$

Se redondea a 2 decimales y se reordenan las columnas en un orden lógico (identificación → producto → métricas), ordenando las filas por fecha.

In [11]:
ventas = etl.agregar_total_venta(ventas)
display(ventas.head(10))
display(ventas[["cantidad", "precio_unitario", "total_venta"]].describe().round(2))

,id_transaccion,fecha,id_producto,nombre_producto,cantidad,precio_unitario,total_venta
0,341,2026-01-02,120,Mochila para notebook,2,570339.64,1140679.28
1,425,2026-01-02,118,Pendrive 64GB,5,62116.44,310582.20
2,805,2026-01-02,115,Tablet 10'',5,401855.71,2009278.55
3,806,2026-01-02,113,Hub USB-C,5,581259.28,2906296.40
4,861,2026-01-02,111,Silla ergonómica,6,336864.23,2021185.38
5,64,2026-01-03,104,Sin nombre,2,629144.39,1258288.78
6,263,2026-01-03,114,Parlante portátil,6,741371.64,4448229.84
7,535,2026-01-03,103,Teclado mecánico,8,773445.14,6187561.12
8,667,2026-01-03,120,Mochila para notebook,8,570339.64,4562717.12
9,750,2026-01-04,116,Smartwatch,9,208378.66,1875407.94


,cantidad,precio_unitario,total_venta
count,886.00,886.00,886.00
mean,5.49,507701.09,2780115.04
std,2.88,252345.45,2139218.40
min,1.00,62116.44,62116.44
25%,3.00,336864.23,837631.90
50%,5.00,570339.64,2325037.12
75%,8.00,741371.64,4080954.10
max,10.00,878182.00,8781820.00


## 6. Validación del dataset final

`etl.validar()` lanza un error si alguna condición falla:
- Sin nulos en los campos críticos (`id_transaccion`, `fecha`, `id_producto`, `cantidad`, `precio_unitario`, `total_venta`).
- `fecha` de tipo `datetime64`.
- `id_transaccion` único (confirma que el merge no duplicó filas).
- Cantidades y precios positivos.
- `total_venta` igual a `cantidad * precio_unitario`.

In [12]:
etl.validar(ventas)
print("✔ Validación superada\n")

print("Nulos por columna en el dataset final:")
print(ventas.isna().sum())
print()
ventas.info()

total_filas = len(trans_raw)
print(f"\nFilas conservadas: {len(ventas)} de {total_filas} ({len(ventas) / total_filas:.1%})")
print(f"Período: {ventas['fecha'].min():%d/%m/%Y} a {ventas['fecha'].max():%d/%m/%Y}")
print(f"Facturación total: ${ventas['total_venta'].sum():,.2f}")

✔ Validación superada

Nulos por columna en el dataset final:
id_transaccion     0
fecha              0
id_producto        0
nombre_producto    0
cantidad           0
precio_unitario    0
total_venta        0
dtype: int64

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 886 entries, 0 to 885
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   id_transaccion   886 non-null    int64         
 1   fecha            886 non-null    datetime64[ns]
 2   id_producto      886 non-null    int64         
 3   nombre_producto  886 non-null    string        
 4   cantidad         886 non-null    int64         
 5   precio_unitario  886 non-null    float64       
 6   total_venta      886 non-null    float64       
dtypes: datetime64[ns](1), float64(2), int64(3), string(1)
memory usage: 48.6 KB

Filas conservadas: 886 de 1000 (88.6%)
Período: 02/01/2026 a 27/10/2026
Facturación total: $2,463,181,925.31


## 7. Exportación a Parquet

¿Por qué Parquet y no CSV?
- **Columnar y comprimido:** ocupa menos espacio y se leen solo las columnas necesarias.
- **Conserva los tipos:** `fecha` vuelve como `datetime64` y los IDs como enteros, sin tener que reconvertirlos (en un CSV todo vuelve como texto o se infiere).
- Es el formato que se va a usar más adelante con **DuckDB** y **Polars**.

Después de guardar, se vuelve a leer el archivo para comprobar que el contenido y los tipos son idénticos.

In [13]:
etl.exportar_parquet(ventas, PATH_PARQUET)

# Verificación: lectura de vuelta
ventas_parquet = pd.read_parquet(PATH_PARQUET)
pd.testing.assert_frame_equal(ventas, ventas_parquet, check_dtype=False)
print("✔ El Parquet se leyó de vuelta y coincide con el DataFrame en memoria")
print("\nTipos al leer el Parquet:")
print(ventas_parquet.dtypes)

# Comparación de tamaño contra el mismo dataset en CSV
path_csv_tmp = PROCESSED_DIR / "_comparacion.csv"
ventas.to_csv(path_csv_tmp, index=False)
kb_parquet = PATH_PARQUET.stat().st_size / 1024
kb_csv = path_csv_tmp.stat().st_size / 1024
path_csv_tmp.unlink()
print(f"\nTamaño Parquet: {kb_parquet:.1f} KB | mismo dataset en CSV: {kb_csv:.1f} KB "
      f"({1 - kb_parquet / kb_csv:.0%} menos)")

✔ El Parquet se leyó de vuelta y coincide con el DataFrame en memoria

Tipos al leer el Parquet:
id_transaccion              int64
fecha              datetime64[ns]
id_producto                 int64
nombre_producto    string[python]
cantidad                    int64
precio_unitario           float64
total_venta               float64
dtype: object

Tamaño Parquet: 17.0 KB | mismo dataset en CSV: 49.0 KB (65% menos)
